# Kaggle Submit — Smart MCQ Solver (inference-only)
**Roll No:** 23f3004491

This notebook does the **inference/submission** step described in the project guidelines:
heavy compute was done separately (Colab), and the precomputed model scores are imported
here as a dataset. This notebook applies the retrieval lookup + gating and writes the
submission. **Runs on CPU — no GPU needed**, so it commits fast and reliably.

### Setup
1. Run the Colab notebook, download `qwen_scores.csv`.
2. On Kaggle: **Add Input → Upload dataset** → upload `qwen_scores.csv`
   (note the dataset folder name it gets, e.g. `qwen-scores`).
3. Set `SCORES_PATH` below to match, then Run All.

In [1]:
import os
for dirpath, _, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(dirpath, f))

/kaggle/input/datasets/tarungangwar/qwen-scores/qwen_scores.csv
/kaggle/input/datasets/tarungangwar/dl-gen-data/sample_submission.csv
/kaggle/input/datasets/tarungangwar/dl-gen-data/train.csv
/kaggle/input/datasets/tarungangwar/dl-gen-data/test.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import re
import numpy as np
import pandas as pd

COMP = "/kaggle/input/competitions/smart-mcq-solver-challenge"
SCORES_PATH = "/kaggle/input/datasets/tarungangwar/qwen-scores/qwen_scores.csv"

train = pd.read_csv(f"{COMP}/train.csv")
test  = pd.read_csv(f"{COMP}/test.csv")
scores = pd.read_csv(SCORES_PATH)

OPTIONS = ['A', 'B', 'C', 'D', 'E']
print("test", test.shape, "| scores", scores.shape)

test (500, 7) | scores (500, 6)


## 1. Prompt normalization + retrieval lookup (from train duplicates)

In [3]:
WRAPPERS_START = ["pick the best possible answer:", "select the most accurate option:",
    "identify the correct statement:", "determine the correct option:", "choose the right answer:",
    "answer the following:", "choose the correct option:", "select the correct answer:",
    "pick the correct option:", "identify the right option:"]
WRAPPERS_END = ["among the listed options.", "among the listed options", "carefully.", "carefully",
    "from the choices below.", "from the choices below", "from the options given.", "from the options given"]

def normalize_prompt(p):
    p = str(p).strip().lower()
    changed = True
    while changed:
        changed = False
        for w in WRAPPERS_START:
            if p.startswith(w):
                p = p[len(w):].strip(); changed = True
        for w in WRAPPERS_END:
            if p.endswith(w):
                p = p[:-len(w)].strip(); changed = True
    return re.sub(r'\s+', ' ', p)

train['core'] = train['prompt'].apply(normalize_prompt)
test['core']  = test['prompt'].apply(normalize_prompt)
train['answer_text'] = train.apply(lambda r: str(r[r['answer']]), axis=1)
answer_lookup = dict(zip(train['core'], train['answer_text']))

import difflib
def lookup_with_confidence(row):
    if row['core'] not in answer_lookup:
        return None, None
    ans = answer_lookup[row['core']].strip()
    for o in OPTIONS:
        if str(row[o]).strip() == ans:
            return o, 'exact'
    sims = sorted(((difflib.SequenceMatcher(None, str(row[o]).strip().lower(), ans.lower()).ratio(), o)
                   for o in OPTIONS), reverse=True)
    (s1, o1), (s2, _) = sims[0], sims[1]
    if s1 > 0.9 and (s1 - s2) > 0.05:
        return o1, 'fuzzy'
    return None, None

matches = test.apply(lookup_with_confidence, axis=1)
test['lookup_answer'] = [m[0] for m in matches]
test['lookup_kind']   = [m[1] for m in matches]
print(test['lookup_kind'].value_counts(dropna=False).to_string())

lookup_kind
exact    362
None     111
fuzzy     27


## 2. Combine Qwen scores with lookup (lookup-first gating)

In [4]:
score_cols = [f"p_{L}" for L in OPTIONS]
score_map = {row['id']: [row[c] for c in score_cols] for _, row in scores.iterrows()}

def qwen_rank(qid):
    s = score_map[qid]
    order = np.argsort(s)[::-1]
    return [OPTIONS[i] for i in order]

final_preds = []
for _, row in test.iterrows():
    qr = qwen_rank(row['id'])
    la, kind = row['lookup_answer'], row['lookup_kind']
    if kind == 'exact':
        rest = [o for o in qr if o != la]
        final_preds.append([la] + rest[:2])
    elif kind == 'fuzzy':
        top = qr[0]
        if la == top:
            final_preds.append(qr[:3])
        else:
            rest = [o for o in qr if o not in (top, la)]
            final_preds.append([top, la] + rest[:1])
    else:
        final_preds.append(qr[:3])

print("built", len(final_preds), "predictions")

built 500 predictions


## 3. Write and validate submission

In [5]:
submission = pd.DataFrame({
    "ID": test['id'],
    "Prediction": [" ".join(p[:3]) for p in final_preds]
})
submission.to_csv("submission.csv", index=False)

assert list(submission.columns) == ["ID", "Prediction"]
assert submission['Prediction'].str.split().str.len().eq(3).all()
assert len(submission) == len(test)
assert submission['Prediction'].apply(lambda s: all(t in OPTIONS for t in s.split())).all()

print(submission.head())
print("\nRows:", len(submission), "- submission.csv ready (inference-only, CPU).")

   ID Prediction
0   1      A D E
1   2      B A E
2   3      B C D
3   4      E A C
4   5      C D E

Rows: 500 - submission.csv ready (inference-only, CPU).
